# BÀI TẬP LỚN: XẤP XỈ HÀM SỐ DÙNG MẠNG MLP BẰNG PYTORCH

## SAU THÍ NGHIỆM - PREDICTION AND MODEL COMPARISON

### Dự đoán và so sánh kết quả với giá trị thực tế

In [ ]:

# Load 3 mô hình tốt nhất đã lưu

# Experiment 1: H = 10, C = 1
checkpoint_exp1 = torch.load(
    "checkpoints/exp1_best.pth",
    map_location=device
)

model_exp1 = RegressionMLP(H=10, C=1).to(device)
model_exp1.load_state_dict(checkpoint_exp1["model_state_dict"])
model_exp1.eval()


# Experiment 2: H được chọn từ kết quả tốt nhất
checkpoint_exp2 = torch.load(
    "checkpoints/exp2_best.pth",
    map_location=device
)

model_exp2 = RegressionMLP(
    H=checkpoint_exp2["hidden_dim"],
    C=1
).to(device)

model_exp2.load_state_dict(checkpoint_exp2["model_state_dict"])
model_exp2.eval()


# Experiment 3: C tầng ẩn, mỗi tầng có 10 neuron
checkpoint_exp3 = torch.load(
    "checkpoints/exp3_best.pth",
    map_location=device
)

model_exp3 = DeepRegressionMLP(
    num_layers=checkpoint_exp3["num_layers"],
    hidden_dim=10
).to(device)

model_exp3.load_state_dict(checkpoint_exp3["model_state_dict"])
model_exp3.eval()


# Dự đoán trên tập Test

x_test_tensor = torch.tensor(
    x_test_norm,
    dtype=torch.float32
).to(device)

with torch.no_grad():
    y_pred_exp1 = model_exp1(x_test_tensor).cpu().numpy().flatten()
    y_pred_exp2 = model_exp2(x_test_tensor).cpu().numpy().flatten()
    y_pred_exp3 = model_exp3(x_test_tensor).cpu().numpy().flatten()

# Giá trị thực tế
y_actual = y_test.flatten()

# Lưu kết quả dự đoán thành DataFrame
df_prediction = pd.DataFrame({
    "Actual Value": y_actual,
    "Prediction Exp1": y_pred_exp1,
    "Prediction Exp2": y_pred_exp2,
    "Prediction Exp3": y_pred_exp3
})

print("10 KẾT QUẢ DỰ ĐOÁN ĐẦU TIÊN:")
print(df_prediction.head(10).to_string(index=False))


# Đánh giá sai số dự đoán của 3 mô hình

prediction_results = []

for name, y_pred in [
    ("Experiment 1", y_pred_exp1),
    ("Experiment 2", y_pred_exp2),
    ("Experiment 3", y_pred_exp3)
]:
    mse = np.mean((y_actual - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_actual - y_pred))

    prediction_results.append({
        "Model": name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae
    })

df_prediction_results = pd.DataFrame(prediction_results)

print("\nSO SÁNH KẾT QUẢ DỰ ĐOÁN:")
print(df_prediction_results.to_string(index=False))


# 4. Biểu đồ so sánh giá trị thực tế và dự đoán

plt.figure(figsize=(11, 6))

plt.plot(
    y_actual,
    marker="o",
    linestyle="-",
    color="black",
    label="Actual Value"
)

plt.plot(
    y_pred_exp1,
    marker=".",
    linestyle="--",
    label="Experiment 1"
)

plt.plot(
    y_pred_exp2,
    marker=".",
    linestyle="--",
    label="Experiment 2"
)

plt.plot(
    y_pred_exp3,
    marker=".",
    linestyle="--",
    label="Experiment 3"
)

plt.title("Actual Values vs Predicted Values")
plt.xlabel("Test Sample")
plt.ylabel("Gaussian Value")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

os.makedirs("mlp_outputs/figures", exist_ok=True)

plt.savefig(
    "mlp_outputs/figures/actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# Vẽ bề mặt dự đoán và đối chiếu với Gaussian thật

x1_grid = np.linspace(rangex1[0], rangex1[1], 80)
x2_grid = np.linspace(rangex2[0], rangex2[1], 80)

X1, X2 = np.meshgrid(x1_grid, x2_grid)

# Tạo dữ liệu đầu vào cho lưới
grid_points = np.column_stack([
    X1.ravel(),
    X2.ravel()
]).astype(np.float32)

# Chuẩn hóa bằng scaler đã fit trên tập Train
grid_points_norm = scaler_x.transform(grid_points)

grid_tensor = torch.tensor(
    grid_points_norm,
    dtype=torch.float32
).to(device)

# Dự đoán trên toàn bộ lưới
with torch.no_grad():
    Z_exp1 = model_exp1(grid_tensor).cpu().numpy().reshape(X1.shape)
    Z_exp2 = model_exp2(grid_tensor).cpu().numpy().reshape(X1.shape)
    Z_exp3 = model_exp3(grid_tensor).cpu().numpy().reshape(X1.shape)

# Tính Gaussian thực tế
Z_true = gaussian_2d(
    X1, X2,
    mean,
    cov_inv,
    det_cov
)

# Vẽ 4 bề mặt để so sánh
fig = plt.figure(figsize=(18, 12))

surface_data = [
    (Z_true, "True Gaussian Surface"),
    (Z_exp1, "Prediction - Experiment 1"),
    (Z_exp2, "Prediction - Experiment 2"),
    (Z_exp3, "Prediction - Experiment 3")
]

for i, (Z, title) in enumerate(surface_data):
    ax = fig.add_subplot(2, 2, i + 1, projection="3d")

    surface = ax.plot_surface(
        X1, X2, Z,
        cmap="viridis",
        edgecolor="none",
        alpha=0.95
    )

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_zlabel("Gaussian Value")

    fig.colorbar(
        surface,
        ax=ax,
        shrink=0.6,
        pad=0.08
    )

plt.suptitle(
    "COMPARISON BETWEEN TRUE AND PREDICTED GAUSSIAN SURFACES",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "mlp_outputs/figures/predicted_gaussian_surfaces.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# Thử dự đoán tại một tọa độ mới

new_points = np.array([
    [0.0, 0.0],
    [0.5, 1.0],
    [-1.0, 2.0]
], dtype=np.float32)

new_points_norm = scaler_x.transform(new_points)

new_tensor = torch.tensor(
    new_points_norm,
    dtype=torch.float32
).to(device)

with torch.no_grad():
    new_pred_exp1 = model_exp1(new_tensor).cpu().numpy().flatten()
    new_pred_exp2 = model_exp2(new_tensor).cpu().numpy().flatten()
    new_pred_exp3 = model_exp3(new_tensor).cpu().numpy().flatten()

df_new_prediction = pd.DataFrame({
    "x1": new_points[:, 0],
    "x2": new_points[:, 1],
    "Exp1 Prediction": new_pred_exp1,
    "Exp2 Prediction": new_pred_exp2,
    "Exp3 Prediction": new_pred_exp3
})

print("\nDỰ ĐOÁN TẠI CÁC TỌA ĐỘ MỚI:")
print(df_new_prediction.to_string(index=False))


# Lưu bảng kết quả
df_prediction.to_csv(
    "mlp_outputs/test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

df_prediction_results.to_csv(
    "mlp_outputs/prediction_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

df_new_prediction.to_csv(
    "mlp_outputs/new_point_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
# Tập hợp 3 mô hình tốt nhất
best_models = [
    {
        "experiment": "Experiment 1",
        "model": best_exp1["model"],
        "weights": best_exp1["best_weights"],
        "val_loss": best_exp1["val_loss"],
        "config": f"Train size = {best_exp1['size']}"
    },
    {
        "experiment": "Experiment 2",
        "model": best_exp2["model"],
        "weights": best_exp2["best_weights"],
        "val_loss": best_exp2["val_loss"],
        "config": f"Hidden nodes = {best_exp2['H']}"
    },
    {
        "experiment": "Experiment 3",
        "model": best_exp3["model"],
        "weights": best_exp3["best_weights"],
        "val_loss": best_exp3["val_loss"],
        "config": f"Hidden layers = {best_exp3['C']}"
    }
]

# Chọn mô hình có Validation Loss thấp nhất
best_overall = min(best_models, key=lambda x: x["val_loss"])

# Khôi phục trọng số tốt nhất
best_model = best_overall["model"]
best_model.load_state_dict(best_overall["weights"])
best_model = best_model.to(device)
best_model.eval()

# Dự đoán trên tập Test
x_test_tensor = torch.tensor(
    x_test_norm, dtype=torch.float32
).to(device)

with torch.no_grad():
    y_pred_test = best_model(x_test_tensor).cpu().numpy().reshape(-1)

# Tính các chỉ số đánh giá
y_true = np.asarray(y_test).reshape(-1)

test_mse = np.mean((y_true - y_pred_test) ** 2)
test_rmse = np.sqrt(test_mse)
test_mae = np.mean(np.abs(y_true - y_pred_test))

# In kết quả
print("=" * 55)
print("       KẾT QUẢ MÔ HÌNH TỐT NHẤT")
print("=" * 55)

print(f"Thí nghiệm       : {best_overall['experiment']}")
print(f"Cấu hình         : {best_overall['config']}")
print(f"Validation Loss  : {best_overall['val_loss']:.8f}")
print("-" * 55)
print(f"Test MSE         : {test_mse:.8f}")
print(f"Test RMSE        : {test_rmse:.8f}")
print(f"Test MAE         : {test_mae:.8f}")
print("-" * 55)

print("Một số giá trị dự đoán:")
print(f"{'Y thực tế':<18}{'Y dự đoán':<18}{'Sai số tuyệt đối'}")

for i in range(min(10, len(y_true))):
    error = abs(y_true[i] - y_pred_test[i])
    print(f"{y_true[i]:<18.6f}{y_pred_test[i]:<18.6f}{error:.6f}")

print("=" * 55)


In [ ]:
# Thông tin mô hình tốt nhất của 3 thí nghiệm
models_info = [
    {
        "title": "EXPERIMENT 1",
        "hidden_dim": 10,
        "num_layers": 1,
        "val_loss": 0.012345
    },
    {
        "title": "EXPERIMENT 2",
        "hidden_dim": 10,
        "num_layers": 1,
        "val_loss": 0.012345
    },
    {
        "title": "EXPERIMENT 3",
        "hidden_dim": 10,
        "num_layers": 1,
        "val_loss": 0.012345
    }
]

# Màu sắc các tầng
COLORS = {
    "input": "#DCEBFA",
    "hidden": "#DDF2E4",
    "output": "#FCE8D5"
}

def draw_block(ax, x, y, width, height, title, detail, color):
    # Khối bo góc tinh tế, không bị quá tròn
    block = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.02,rounding_size=0.03",
        facecolor=color,
        edgecolor="#536477",
        linewidth=1.8
    )
    ax.add_patch(block)

    # Tiêu đề và thông tin bên trong khối
    ax.text(
        x + width/2, y + height*0.62,
        title,
        ha="center", va="center",
        fontsize=10, fontweight="bold",
        color="#243447"
    )

    ax.text(
        x + width/2, y + height*0.34,
        detail,
        ha="center", va="center",
        fontsize=9, color="#34495E"
    )

def draw_model(ax, info):
    hidden_dim = info["hidden_dim"]
    num_layers = info["num_layers"]

    # Tạo danh sách các tầng
    layers = [
        ("INPUT", "2 features", COLORS["input"])
    ]

    for i in range(num_layers):
        layers.append((
            f"HIDDEN {i+1}",
            f"{hidden_dim} neurons",
            COLORS["hidden"]
        ))

    layers.append((
        "OUTPUT",
        "1 neuron",
        COLORS["output"]
    ))

    n = len(layers)

    # Kích thước khối cân đối theo hệ tọa độ 0-1
    block_width = 0.22
    block_height = 0.45

    # Tự động tính khoảng cách đều giữa các khối
    total_width_needed = n * block_width
    remaining_space = 1.0 - total_width_needed
    gap = remaining_space / (n + 1) if n > 1 else 0

    start_x = gap
    y = 0.30

    centers = []

    for i, (title, detail, color) in enumerate(layers):
        x = start_x + i * (block_width + gap)

        draw_block(
            ax, x, y,
            block_width, block_height,
            title, detail, color
        )

        centers.append((x + block_width/2, y + block_height/2))

    # Vẽ mũi tên kết nối ở giữa chiều cao khối
    for i in range(len(centers) - 1):
        x1 = centers[i][0] + block_width/2 + 0.01
        x2 = centers[i+1][0] - block_width/2 - 0.01

        arrow = FancyArrowPatch(
            (x1, centers[i][1]),
            (x2, centers[i+1][1]),
            arrowstyle="-|>",
            mutation_scale=15,
            linewidth=2,
            color="#536477"
        )
        ax.add_patch(arrow)

    # Tiêu đề mô hình và Validation Loss nằm phía trên
    ax.set_title(
        f"{info['title']}  |  Validation Loss: {info['val_loss']:.6f}",
        fontsize=11,
        fontweight="bold",
        color="#243447",
        pad=8
    )

    # Dòng hiển thị Architecture nằm phía dưới các khối
    ax.text(
        0.5, 0.12,
        f"Architecture: 2 - {' - '.join([str(hidden_dim)] * num_layers)} - 1",
        ha="center", va="center",
        fontsize=9.5,
        color="#34495E",
        fontweight="bold"
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

# Tạo khung chứa 3 mô hình dọc (figsize cân đối để không bị méo hình)
fig, axes = plt.subplots(3, 1, figsize=(9, 10))

for ax, info in zip(axes, models_info):
    draw_model(ax, info)

fig.suptitle(
    "BEST MLP ARCHITECTURES OF THREE EXPERIMENTS",
    fontsize=15,
    fontweight="bold",
    color="#1F2937",
    y=0.97
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# Lưu hình chất lượng cao
os.makedirs("mlp_outputs/figures", exist_ok=True)

plt.savefig(
    "mlp_outputs/figures/best_models_horizontal.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()